# SaansCare – Model 1: AQI Forecasting Pipeline
**Goal:** predict tomorrow's (and next week's) PM2.5 / AQI for each Lahore zone from the last two weeks of pollution and weather.
**Algorithms:** baselines (persistence, 7-day mean, Ridge), Random Forest, **XGBoost** (main, tuned), **Prophet** (trend/seasonality baseline).
**Metrics:** RMSE, MAE, R², and AQI-category accuracy. **Split:** chronological (never shuffled).

**How to run:** run the cells top to bottom. Step 2 will ask you for the dataset file (`aqi_forecasting.csv` / `.xlsx`).
Works in Google Colab (upload dialog) and in VS Code / Jupyter (asks for the file path).

## 1. Setup
*What we are doing:* importing the libraries and creating an output folder. Missing packages are installed automatically.

In [ ]:
import sys, subprocess, os, json, warnings, zipfile, time
from pathlib import Path
warnings.filterwarnings("ignore")

def _ensure(pkg, import_name=None):
    """Install a package only if it is missing (Google Colab already has most of them)."""
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_ensure("xgboost"); _ensure("openpyxl"); _ensure("joblib")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
                             classification_report, mean_squared_error, mean_absolute_error, r2_score)
from xgboost import XGBClassifier, XGBRegressor

RANDOM_STATE = 42                      # fixed seed -> reproducible results
np.random.seed(RANDOM_STATE)
OUT_DIR = Path("saanscare_outputs/model1_aqi_forecasting")   # everything this notebook produces is saved here
OUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})
print("Setup complete. Outputs will be saved in:", OUT_DIR.resolve())

In [ ]:
def plot_confusion(y_true, y_pred, labels, title, path=None):
    """Confusion matrix as counts and as row-percentages (recall per class)."""
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))
    cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(5.8, 5))
    ax.imshow(cmn, cmap="Greens", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{cm[i, j]:,}\n{cmn[i, j]*100:.0f}%", ha="center", va="center",
                    color="white" if cmn[i, j] > 0.55 else "black", fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(title); ax.grid(False)
    plt.tight_layout()
    if path: plt.savefig(path, dpi=130)
    plt.show()

def warning_rates(y_true, y_pred):
    """For ordered risk classes: under-warning (predicted LOWER risk than reality) is the dangerous error."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return {"exact": float((y_pred == y_true).mean()),
            "under_warning": float((y_pred < y_true).mean()),
            "over_warning": float((y_pred > y_true).mean())}

def clf_scores(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0)}

def build_pre(num_cols, cat_cols, scale=True):
    """Preprocessing: impute missing values, scale numbers, one-hot encode text categories."""
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale: num_steps.append(("sc", StandardScaler()))
    cat_steps = [("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]
    return ColumnTransformer([("num", Pipeline(num_steps), num_cols), ("cat", Pipeline(cat_steps), cat_cols)])

def to_jsonable(o):
    if isinstance(o, dict): return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_jsonable(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return o

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(to_jsonable(obj), f, indent=2)

def package_outputs(out_dir, zip_name):
    """Zip the output folder; in Google Colab this also starts the download."""
    zpath = Path(zip_name)
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for p in Path(out_dir).rglob("*"):
            if p.is_file(): z.write(p, p.relative_to(Path(out_dir).parent))
    print("Packaged:", zpath.resolve(), f"({zpath.stat().st_size/1e6:.1f} MB)")
    try:
        from google.colab import files
        files.download(str(zpath))
    except ImportError:
        pass

## 2. Load the dataset (the notebook asks you for it)
*What we are doing:* asking you for the dataset file, reading it, and checking that all expected columns exist.

In [ ]:
def load_dataset(label, required_cols):
    """Ask the user for the dataset, read it, and check that the expected columns are present."""
    path = os.environ.get("SAANSCARE_DATASET")            # optional shortcut: skips the prompt
    if not path:
        try:
            from google.colab import files                 # running in Google Colab -> upload dialog
            print(f"Upload your {label} file (.csv or .xlsx) ...")
            uploaded = files.upload()
            if not uploaded:
                raise FileNotFoundError("No file was uploaded.")
            path = next(iter(uploaded))
        except ImportError:                                # running in VS Code / Jupyter -> ask for the path
            path = input(f"Enter the full path of your {label} file (.csv or .xlsx): ").strip().strip('"').strip("'")
    df = pd.read_excel(path) if path.lower().endswith((".xlsx", ".xls")) else pd.read_csv(path)
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"The file is missing these required columns: {missing}\nColumns found: {list(df.columns)}")
    print(f"Loaded {Path(path).name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    return df

REQUIRED = ["date", "zone", "pm25", "pm10", "no2", "so2", "co", "o3", "temperature", "humidity",
            "wind_speed", "wind_dir", "pressure", "rainfall"]
df = load_dataset("AQI forecasting dataset", REQUIRED)
df.head()

## 3. Data understanding (EDA)
*What we are doing:* looking at the data before touching it: size, types, missing values, date range, and the seasonal pattern.

In [ ]:
df["date"] = pd.to_datetime(df["date"])
print("Date range:", df["date"].min().date(), "->", df["date"].max().date())
print("Zones:", df["zone"].nunique(), "|", sorted(df["zone"].unique()))
print("\nMissing values (%):")
print((df.isna().mean() * 100).round(2)[lambda s: s > 0].to_string())
display(df.describe().T.round(2))

monthly = df.groupby([df["date"].dt.year.rename("year"), df["date"].dt.month.rename("month")])["pm25"].mean().unstack(0)
monthly.plot(marker="o", title="Monthly mean PM2.5 by year (ug/m3)")
plt.xlabel("month"); plt.ylabel("PM2.5"); plt.show()

## 4. Data cleaning
*What we are doing:* fixing problems that would mislead the model: duplicate rows, missing days, sensor spikes and gaps.
Spikes are detected as values far above the local 7-day median, removed, and short gaps are filled by interpolation. AQI is then recomputed from PM2.5.

In [ ]:
POLL = ["pm25", "pm10", "no2", "so2", "co", "o3"]
WEATHER = ["temperature", "humidity", "wind_speed", "wind_dir", "pressure", "rainfall"]

def pm25_to_aqi(pm):
    """US EPA PM2.5 -> AQI (2024 breakpoints)."""
    bp = [(0.0, 9.0, 0, 50), (9.1, 35.4, 51, 100), (35.5, 55.4, 101, 150),
          (55.5, 125.4, 151, 200), (125.5, 225.4, 201, 300), (225.5, 325.4, 301, 500)]
    pm = np.asarray(pm, float); out = np.full(pm.shape, np.nan); c_ = np.floor(pm * 10) / 10
    for lo, hi, ilo, ihi in bp:
        m = (c_ >= lo) & (c_ <= hi)
        out[m] = (ihi - ilo) / (hi - lo) * (c_[m] - lo) + ilo
    out[c_ > 325.4] = 500
    return np.round(out)

def aqi_band(aqi):
    """0=Good ... 5=Hazardous"""
    return np.digitize(np.asarray(aqi, float), [50, 100, 150, 200, 300])

BAND_NAMES = ["Good", "Moderate", "Unhealthy for Sensitive Groups", "Unhealthy", "Very Unhealthy", "Hazardous"]

n0 = len(df)
df = df.drop_duplicates(["date", "zone"]).sort_values(["zone", "date"]).reset_index(drop=True)
print("Duplicates removed:", n0 - len(df))

# make every zone a complete daily calendar so that gaps become visible (NaN) instead of silently missing
parts = []
for z, g in df.groupby("zone"):
    g = g.set_index("date").reindex(pd.date_range(g["date"].min(), g["date"].max(), freq="D"))
    g["zone"] = z; g.index.name = "date"
    parts.append(g.reset_index())
df = pd.concat(parts, ignore_index=True)
for col in ["lat", "lon"]:
    if col in df: df[col] = df.groupby("zone")[col].transform(lambda s: s.ffill().bfill())
print("Rows after completing the calendar:", len(df))

# sensor spikes: far above the local median
med7 = df.groupby("zone")["pm25"].transform(lambda s: s.rolling(7, center=True, min_periods=3).median())
spike = (df["pm25"] > 2.2 * med7) & ((df["pm25"] - med7) > 40)
print("Spikes detected:", int(spike.sum()))
if "quality_flag" in df:   # only present in the SaansCare synthetic set: check our detector against it
    print(pd.crosstab(df["quality_flag"], spike, rownames=["flag in file"], colnames=["detected"]))
df.loc[spike, POLL] = np.nan

before = df[POLL + WEATHER].isna().sum().sum()
for col in POLL + WEATHER:
    if col == "rainfall":
        df[col] = df[col].fillna(0)
    else:
        df[col] = df.groupby("zone")[col].transform(lambda s: s.interpolate(limit=7, limit_direction="both"))
print("Missing cells before/after interpolation:", before, "->", df[POLL + WEATHER].isna().sum().sum())

df["aqi"] = pm25_to_aqi(df["pm25"])          # recompute so corrupted AQI values never reach the model
df.head()

## 5. Feature engineering
*What we are doing:* turning each day into a row of "what we know today" so the model can predict `h` days ahead:
yesterday's and last week's PM2.5 (lags), recent averages (rolling means), today's weather, and calendar features
(month and weekday as smooth sin/cos waves). The target is PM2.5 `h` days in the future.

In [ ]:
def build_features(data, h, keep_unlabeled=False):
    """One row per (zone, day). Target = PM2.5 h days later. Only past information is used (no leakage)."""
    out = []
    for z, d in data.sort_values(["zone", "date"]).groupby("zone"):
        d = d.copy()
        for L in (1, 2, 3, 7, 14):
            d[f"pm25_lag{L}"] = d["pm25"].shift(L)
        for W in (3, 7, 14):
            d[f"pm25_roll{W}"] = d["pm25"].shift(1).rolling(W).mean()
        d["pm25_std7"] = d["pm25"].shift(1).rolling(7).std()
        d["pm25_diff1"] = d["pm25"] - d["pm25"].shift(1)
        d["target_pm25"] = d["pm25"].shift(-h)
        out.append(d)
    f = pd.concat(out, ignore_index=True)
    f["dow"] = f["date"].dt.dayofweek
    f["is_weekend"] = (f["dow"] >= 5).astype(int)
    f["month_sin"] = np.sin(2 * np.pi * f["date"].dt.month / 12); f["month_cos"] = np.cos(2 * np.pi * f["date"].dt.month / 12)
    f["doy_sin"] = np.sin(2 * np.pi * f["date"].dt.dayofyear / 365.25); f["doy_cos"] = np.cos(2 * np.pi * f["date"].dt.dayofyear / 365.25)
    f["wd_sin"] = np.sin(np.radians(f["wind_dir"])); f["wd_cos"] = np.cos(np.radians(f["wind_dir"]))
    f = f.sort_values(["date", "zone"]).reset_index(drop=True)
    need = FEATURES + ([] if keep_unlabeled else ["target_pm25"])
    return f.dropna(subset=need).reset_index(drop=True)

NUM_FEATURES = (["pm25", "pm10", "no2", "so2", "co", "o3", "temperature", "humidity", "wind_speed", "pressure", "rainfall",
                 "wd_sin", "wd_cos", "pm25_lag1", "pm25_lag2", "pm25_lag3", "pm25_lag7", "pm25_lag14",
                 "pm25_roll3", "pm25_roll7", "pm25_roll14", "pm25_std7", "pm25_diff1",
                 "dow", "is_weekend", "month_sin", "month_cos", "doy_sin", "doy_cos"])
CAT_FEATURES = ["zone"]
FEATURES = NUM_FEATURES + CAT_FEATURES

feat1 = build_features(df, h=1)
print("Rows for 1-day-ahead model:", len(feat1), "| features:", len(FEATURES))
feat1[["date", "zone", "pm25", "pm25_lag1", "pm25_roll7", "target_pm25"]].head()

## 6. Train / validation / test split (chronological)
*What we are doing:* splitting by **time** (70% oldest days = train, next 15% = validation, newest 15% = test).
A random split would let the model "see the future" through neighbouring days and give fake-good scores.

In [ ]:
# OPTIONAL: choose the split dates yourself, e.g. TRAIN_END = "2025-06-30", VAL_END = "2025-10-31" to put a smog season in the test set.
# Leave both as None to use the default 70% / 15% / 15% chronological split.
TRAIN_END, VAL_END = None, None

def time_split(feat, train_frac=0.70, val_frac=0.15):
    dates = np.sort(feat["date"].unique())
    d_tr = np.datetime64(TRAIN_END) + np.timedelta64(1, "D") if TRAIN_END else dates[int(len(dates) * train_frac)]
    d_va = np.datetime64(VAL_END) + np.timedelta64(1, "D") if VAL_END else dates[int(len(dates) * (train_frac + val_frac))]
    tr = feat[feat["date"] < d_tr]; va = feat[(feat["date"] >= d_tr) & (feat["date"] < d_va)]; te = feat[feat["date"] >= d_va]
    return tr, va, te

train, val, test = time_split(feat1)
for name, part in [("train", train), ("validation", val), ("test", test)]:
    print(f"{name:10s} {part['date'].min().date()} -> {part['date'].max().date()}  ({len(part):,} rows)")
winter_in_test = test["date"].dt.month.isin([11, 12, 1, 2]).mean()
if winter_in_test < 0.2:
    print("\nNOTE: the test period contains little smog season. Report this limitation, or move the split dates so that a winter is in the test set.")
Xtr, ytr, Xva, yva, Xte, yte = train[FEATURES], train["target_pm25"], val[FEATURES], val["target_pm25"], test[FEATURES], test["target_pm25"]

## 7. Preprocessing pipeline
*What we are doing:* building one reusable transformer: fill missing numbers (median), scale numbers, one-hot encode `zone`.
It is saved together with the model so the web app applies exactly the same steps.

In [ ]:
pre_scaled = build_pre(NUM_FEATURES, CAT_FEATURES, scale=True)     # for linear models
pre_trees = build_pre(NUM_FEATURES, CAT_FEATURES, scale=False)     # trees do not need scaling

def score(y_true, y_pred):
    """RMSE / MAE / R2 on PM2.5 and accuracy of the resulting AQI category."""
    y_pred = np.clip(y_pred, 0, None)
    return {"RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))), "MAE": float(mean_absolute_error(y_true, y_pred)),
            "R2": float(r2_score(y_true, y_pred)),
            "AQI_category_acc": float((aqi_band(pm25_to_aqi(y_true)) == aqi_band(pm25_to_aqi(y_pred))).mean())}
print("Preprocessing ready")

## 8. Baselines and model training
*What we are doing:* training several models and comparing them on the validation set. Baselines show the minimum a "real" model must beat:
**persistence** (tomorrow = today) and the **7-day mean**. Then Ridge, Random Forest and XGBoost.

In [ ]:
results = {}
results["Persistence (today)"] = score(yva, Xva["pm25"].values)
results["7-day mean"] = score(yva, Xva["pm25_roll7"].values)

models = {
    "Ridge": Pipeline([("pre", pre_scaled), ("model", Ridge(alpha=1.0))]),
    "Random Forest": Pipeline([("pre", pre_trees), ("model", RandomForestRegressor(n_estimators=250, min_samples_leaf=3, n_jobs=-1, random_state=RANDOM_STATE))]),
    "XGBoost": Pipeline([("pre", pre_trees), ("model", XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.85,
                                                                    colsample_bytree=0.85, tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE))]),
}
for name, pipe in models.items():
    t0 = time.time(); pipe.fit(Xtr, ytr)
    results[name] = score(yva, pipe.predict(Xva))
    print(f"trained {name:14s} in {time.time()-t0:.1f}s")
cmp0 = pd.DataFrame(results).T.round(3).sort_values("RMSE")
print("Best on validation so far:", cmp0.index[0], "(XGBoost stays the main model; if a simpler model wins, say so honestly in your report)")
cmp0

## 9. Prophet baseline (optional)
*What we are doing:* Prophet is a trend + seasonality forecaster. It uses only the date, so we fit it on one zone's training days and forecast the validation
period as a pure time-series baseline. It is expected to be weaker than XGBoost because it cannot see yesterday's pollution or the weather. Skipped automatically if Prophet is not installed.

In [ ]:
try:
    from prophet import Prophet
    import logging; logging.getLogger("prophet").setLevel(logging.ERROR); logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
    zone0 = sorted(df["zone"].unique())[0]
    ptr = train[train["zone"] == zone0][["date", "pm25"]].rename(columns={"date": "ds", "pm25": "y"})
    pva = val[val["zone"] == zone0]
    m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
    m.fit(ptr)
    fc = m.predict(pd.DataFrame({"ds": pva["date"]}))
    prophet_scores = score(pva["target_pm25"].values, fc["yhat"].values)
    xgb_same_zone = score(pva["target_pm25"].values, models["XGBoost"].predict(pva[FEATURES]))
    print(f"Zone: {zone0}")
    print("Prophet :", {k: round(v, 3) for k, v in prophet_scores.items()})
    print("XGBoost :", {k: round(v, 3) for k, v in xgb_same_zone.items()})
    results["Prophet (zone %s only)" % zone0] = prophet_scores
except Exception as e:
    print("Prophet skipped:", e)

## 10. Hyperparameter tuning (XGBoost)
*What we are doing:* searching for better XGBoost settings (tree depth, learning rate, number of trees, ...) using **time-ordered** cross-validation on the training data only.

In [ ]:
def tune_xgb(X, y, n_iter=12):
    base = Pipeline([("pre", pre_trees), ("model", XGBRegressor(tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE))])
    space = {"model__n_estimators": [200, 400, 600], "model__max_depth": [3, 4, 5, 6, 8], "model__learning_rate": [0.03, 0.05, 0.1],
             "model__subsample": [0.7, 0.85, 1.0], "model__colsample_bytree": [0.7, 0.85, 1.0],
             "model__min_child_weight": [1, 3, 5], "model__reg_lambda": [1, 3, 10]}
    order = np.argsort(X.index.values, kind="stable")   # rows are already sorted by date; keep that order for TimeSeriesSplit
    search = RandomizedSearchCV(base, space, n_iter=n_iter, cv=TimeSeriesSplit(3), scoring="neg_root_mean_squared_error",
                                random_state=RANDOM_STATE, n_jobs=1)
    search.fit(X.iloc[order], y.iloc[order])
    return search

t0 = time.time()
search = tune_xgb(Xtr, ytr)
print(f"Search finished in {time.time()-t0:.0f}s. Best CV RMSE: {-search.best_score_:.2f}")
print("Best parameters:", {k.replace("model__", ""): v for k, v in search.best_params_.items()})
best_xgb = search.best_estimator_
results["XGBoost (tuned)"] = score(yva, best_xgb.predict(Xva))
comparison = pd.DataFrame(results).T.round(3).sort_values("RMSE")
comparison

## 11. Final evaluation on the test set
*What we are doing:* the honest score. The tuned XGBoost is retrained on train + validation and evaluated **once** on the untouched, newest test period.

In [ ]:
final_xgb = clone(best_xgb)
trval = pd.concat([train, val])
final_xgb.fit(trval[FEATURES], trval["target_pm25"])
pred_te = np.clip(final_xgb.predict(Xte), 0, None)

test_results = {"Persistence (today)": score(yte, Xte["pm25"].values), "7-day mean": score(yte, Xte["pm25_roll7"].values),
                "XGBoost (final)": score(yte, pred_te)}
print(pd.DataFrame(test_results).T.round(3))

per_zone = test.assign(pred=pred_te).groupby("zone").apply(lambda g: pd.Series(score(g["target_pm25"], g["pred"])))
print("\nPer-zone test performance:"); display(per_zone.round(2))

z = sorted(test["zone"].unique())[0]
g = test.assign(pred=pred_te); g = g[g["zone"] == z]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
ax[0].plot(g["date"], g["target_pm25"], label="actual"); ax[0].plot(g["date"], g["pred"], label="predicted", alpha=.85)
ax[0].set_title(f"Next-day PM2.5 – {z} (test period)"); ax[0].legend()
ax[1].scatter(test["target_pm25"], pred_te, s=6, alpha=.4); lim = [0, max(test["target_pm25"].max(), pred_te.max())]
ax[1].plot(lim, lim, "r--"); ax[1].set_xlabel("actual"); ax[1].set_ylabel("predicted"); ax[1].set_title("Predicted vs actual (all zones)")
plt.tight_layout(); plt.savefig(OUT_DIR / "test_predictions_h1.png", dpi=130); plt.show()

## 12. Feature importance
*What we are doing:* checking which inputs the model relies on (it should rely on recent PM2.5 and weather – which makes physical sense).

In [ ]:
pre_fitted = final_xgb.named_steps["pre"]; names = pre_fitted.get_feature_names_out()
imp = pd.Series(final_xgb.named_steps["model"].feature_importances_, index=names).sort_values(ascending=False).head(15)
imp[::-1].plot(kind="barh", title="Top 15 features (XGBoost gain importance)", color="#16A34A"); plt.tight_layout()
plt.savefig(OUT_DIR / "feature_importance_h1.png", dpi=130); plt.show()

## 13. Multi-day forecast model (7 days ahead)
*What we are doing:* the app also needs a weekly forecast. We repeat the same steps for `h = 7` (reusing the functions above) and compare to the baselines.

In [ ]:
def train_horizon(h, n_iter=8):
    f = build_features(df, h)
    tr, va, te = time_split(f)
    s = tune_xgb(tr[FEATURES], tr["target_pm25"], n_iter=n_iter)
    mdl = clone(s.best_estimator_)
    both = pd.concat([tr, va]); mdl.fit(both[FEATURES], both["target_pm25"])
    p = np.clip(mdl.predict(te[FEATURES]), 0, None)
    res = {"XGBoost": score(te["target_pm25"], p), "Persistence": score(te["target_pm25"], te["pm25"].values),
           "7-day mean": score(te["target_pm25"], te["pm25_roll7"].values)}
    return mdl, res, (tr, va, te)

model_h7, res_h7, _ = train_horizon(7)
print("7-day-ahead test results"); print(pd.DataFrame(res_h7).T.round(3))

## 14. Save the models
*What we are doing:* storing each trained pipeline (preprocessing + model) with its feature list and metrics in one `.joblib` file.
The FastAPI service loads these files; the app never retrains.

In [ ]:
bundles = {
    1: {"pipeline": final_xgb, "metrics": test_results["XGBoost (final)"], "baseline_metrics": {k: v for k, v in test_results.items() if k != "XGBoost (final)"}},
    7: {"pipeline": model_h7, "metrics": res_h7["XGBoost"], "baseline_metrics": {k: v for k, v in res_h7.items() if k != "XGBoost"}},
}
for h, b in bundles.items():
    meta = {"model": "AQI forecasting (XGBoost)", "horizon_days": h, "features": FEATURES, "num_features": NUM_FEATURES, "cat_features": CAT_FEATURES,
            "target": "pm25 (ug/m3) at t+%d; convert to AQI with EPA breakpoints" % h, "zones": sorted(df["zone"].unique()),
            "trained_on_dates": [str(df["date"].min().date()), str(df["date"].max().date())], "metrics_test": b["metrics"],
            "baselines_test": b["baseline_metrics"], "version": "1.0", "random_state": RANDOM_STATE}
    joblib.dump({"pipeline": b["pipeline"], "meta": meta}, OUT_DIR / f"aqi_forecast_h{h}.joblib")
    save_json(meta, OUT_DIR / f"aqi_forecast_h{h}_metadata.json")
save_json({"validation_comparison_h1": results, "test_h1": test_results, "test_h7": res_h7}, OUT_DIR / "all_metrics.json")
comparison.to_csv(OUT_DIR / "model_comparison_validation_h1.csv")
print("Saved:", sorted(p.name for p in OUT_DIR.iterdir()))

## 15. Inference demo (how the web app will use the saved model)
*What we are doing:* loading the saved file and forecasting tomorrow's and next week's AQI for every zone from the most recent data – exactly what the API will do.

In [ ]:
def forecast_latest(df_clean, horizons=(1, 7)):
    rows = []
    for h in horizons:
        bundle = joblib.load(OUT_DIR / f"aqi_forecast_h{h}.joblib")
        f = build_features(df_clean, h, keep_unlabeled=True)
        last = f.sort_values("date").groupby("zone").tail(1)
        pm = np.clip(bundle["pipeline"].predict(last[bundle["meta"]["features"]]), 0, None)
        aqi = pm25_to_aqi(pm)
        rows.append(pd.DataFrame({"zone": last["zone"].values, "based_on_date": last["date"].dt.date.values, "horizon_days": h,
                                  "pred_pm25": pm.round(1), "pred_aqi": aqi, "category": [BAND_NAMES[i] for i in aqi_band(aqi)]}))
    return pd.concat(rows).sort_values(["horizon_days", "pred_aqi"], ascending=[True, False])

forecast_latest(df)

## 16. Package and download
*What we are doing:* zipping every output (models, metadata, metrics, plots) so you can download it and put the models in `ml-service/models/`.

In [ ]:
package_outputs(OUT_DIR, "model1_aqi_forecasting_outputs.zip")

### Notes for the viva
* The split is chronological, features use only past data, and the baselines (persistence, 7-day mean) are reported next to XGBoost.
* If you trained on the SaansCare **synthetic** dataset, the scores show the pipeline works; retrain on the **real** downloaded data (Open-Meteo/CAMS, aqicn, OpenAQ) before claiming real-world accuracy.
* Weather features are today's observations; in production, replace them with the weather *forecast* for the target day to improve the multi-day model.